# NLP: Lab 9 — doc2vec

Використовуємо датасет **AG News** для навчання моделі **doc2vec** та демонстрації семантичного пошуку документів.

**doc2vec** — розширення word2vec для *document embeddings*. Є два варіанти:
- **Distributed Memory (DM)** — навчається передбачати слово за контекстом + вектором документа
- **Distributed Bag-of-Words (DBOW)** — передбачає слова лише з вектора документа

Архітектура має два основних компоненти:
- **projection layer** — формує вектори слів і документів
- **output layer** — розподілена репрезентація, що передбачає цільове слово

## 1. Завантаження датасету AG News

In [1]:
from datasets import load_dataset

ds = load_dataset("fancyzhx/ag_news")
print(ds)
print("\nПриклад запису:")
print(ds['train'][0])

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

Приклад запису:
{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}


## 2. Попередня обробка (Preprocessing)

In [2]:
import pandas as pd
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from gensim.utils import simple_preprocess

label_names = {0: 'World', 1: 'Sports', 2: 'Business', 3: 'Sci/Tech'}

train_df = ds['train'].to_pandas()
test_df  = ds['test'].to_pandas()

print(f"Train: {len(train_df)} рядків, Test: {len(test_df)} рядків")
print()
print(train_df.head(3))

Train: 120000 рядків, Test: 7600 рядків

                                                text  label
0  Wall St. Bears Claw Back Into the Black (Reute...      2
1  Carlyle Looks Toward Commercial Aerospace (Reu...      2
2  Oil and Economy Cloud Stocks' Outlook (Reuters...      2


In [3]:
def tokenize(text):
    return simple_preprocess(str(text))

# TaggedDocument: words = токени, tags = унікальний ID документа
train_corpus = [
    TaggedDocument(words=tokenize(row['text']), tags=[i])
    for i, row in train_df.iterrows()
]

test_corpus = [tokenize(row['text']) for _, row in test_df.iterrows()]

print(f"Train corpus: {len(train_corpus)} документів")
print(f"Test corpus:  {len(test_corpus)} документів")
print()
print("Приклад TaggedDocument:")
print(train_corpus[0])

Train corpus: 120000 документів
Test corpus:  7600 документів

Приклад TaggedDocument:
TaggedDocument<['wall', 'st', 'bears', 'claw', 'back', 'into', 'the', 'black', 'reuters', 'reuters', 'short', 'sellers', 'wall', 'street', 'dwindling', 'band', 'of', 'ultra', 'cynics', 'are', 'seeing', 'green', 'again'], [0]>


## 3. Створення та навчання моделі

In [4]:
import gensim.models

model = gensim.models.doc2vec.Doc2Vec(
    vector_size=50,
    min_count=2,
    epochs=40
)

# Побудова словника
model.build_vocab(train_corpus)
print(f"Розмір словника: {len(model.wv)} слів")
print("Словник доступний через: model.wv")

Розмір словника: 42212 слів
Словник доступний через: model.wv


In [5]:
# Навчання
model.train(
    train_corpus,
    total_examples=model.corpus_count,
    epochs=model.epochs
)
print("Навчання завершено!")
print("\nВектори документів зберігаються в model.dv:")
print(model.dv)

Навчання завершено!

Вектори документів зберігаються в model.dv:
KeyedVectors<vector_size=50, 120000 keys>


## 4. Використання моделі

За допомогою `model.infer_vector` отримуємо вектор для **будь-якого** тексту.

In [6]:
# Інферуємо вектор
sample_words = tokenize('oil price economy market stock')
vector = model.infer_vector(sample_words)
print("Вектор для 'oil price economy market stock':")
print(vector)

Вектор для 'oil price economy market stock':
[ 0.7544387  -0.38034788  0.26799804 -0.48307535 -0.14063309  0.12309655
 -0.05803296 -0.76040035 -0.75083506 -0.5699481   0.8449545   0.2582154
  0.1991885   0.87901586  0.66984427 -0.80662507 -0.50682193 -0.638504
 -0.5368937   0.68362063 -0.25159127 -0.01909491 -0.08368613  0.39845398
  0.5099942   0.32923204 -0.56406534 -0.32087007 -0.6158275   0.7444906
 -0.47452754 -0.5515326  -0.58478254  0.18985625  1.1022329  -0.4147999
 -0.15503557  0.7734391  -0.48783106 -0.7648327  -0.24382459  0.82526743
 -0.27202818  0.04824256 -0.0935135   0.16203062 -1.047878   -0.4836136
 -0.03088121  0.14634995]


In [7]:
# Порівняємо семантичну схожість між текстами
import numpy as np
from gensim import matutils

def cosine_sim(v1, v2):
    return np.dot(matutils.unitvec(v1), matutils.unitvec(v2))

vec_economy = model.infer_vector(tokenize('oil price economy market stock'))
vec_sport   = model.infer_vector(tokenize('soccer football goal championship'))
vec_oil     = model.infer_vector(tokenize('oil gas petroleum barrel refinery'))

print(f"'oil economy' vs 'soccer football' : {cosine_sim(vec_economy, vec_sport):.4f}")
print(f"'oil economy' vs 'oil gas'          : {cosine_sim(vec_economy, vec_oil):.4f}")
print("(вищий = більш семантично схожі)")

'oil economy' vs 'soccer football' : 0.4406
'oil economy' vs 'oil gas'          : 0.7402
(вищий = більш семантично схожі)


## 5. Тестування моделі

Перевіряємо self-similarity: документ повинен знаходити сам себе серед найближчих сусідів.

In [8]:
import collections

ranks = []
second_ranks = []

for doc_id in range(250):
    inferred_vector = model.infer_vector(train_corpus[doc_id].words)
    sims = model.dv.most_similar([inferred_vector], topn=len(model.dv))
    rank = [docid for docid, sim in sims].index(doc_id)
    ranks.append(rank)
    second_ranks.append(sims[1])

counter = collections.Counter(ranks)
print("Розподіл рангів (0 = документ знайшов сам себе):")
print(counter)
print(f"\n% документів що на 1-му місці: {counter[0]/250*100:.1f}%")

Розподіл рангів (0 = документ знайшов сам себе):
Counter({0: 244, 1: 5, 2: 1})

% документів що на 1-му місці: 97.6%


In [9]:
import random

doc_id = random.randint(0, len(test_df) - 1)
test_row = test_df.iloc[doc_id]

inferred_vector = model.infer_vector(test_corpus[doc_id])
sims = model.dv.most_similar([inferred_vector], topn=5)

print(f"=== ТЕСТОВИЙ ДОКУМЕНТ (id={doc_id}) ===")
print(f"Категорія: {label_names[test_row['label']]}")
print(f"Текст: {test_row['text'][:250]}")
print()
print("=== НАЙБІЛЬШ СХОЖІ з тренувального набору ===")
for rank, (train_id, sim) in enumerate(sims[:3], 1):
    train_row = train_df.iloc[train_id]
    match = '✓' if train_row['label'] == test_row['label'] else '✗'
    print(f"[{rank}] sim={sim:.4f} {match} | {label_names[train_row['label']]}")
    print(f"    {train_row['text'][:150]}...")
    print()

=== ТЕСТОВИЙ ДОКУМЕНТ (id=2654) ===
Категорія: Business
Текст: INDUSTRY REPORT: Gambling -- Casinos to be sold Harrah #39;s Entertainment Inc. and Caesars Entertainment Inc. agreed to sell four casino hotels to an affiliate of Colony Capital LLC for about \$1.

=== НАЙБІЛЬШ СХОЖІ з тренувального набору ===
[1] sim=0.7184 ✓ | Business
    Harrah #39;s, Caesars to sell four casinos to satisfy regulators Harrah #39;s Entertainment and Caesars Entertainment Inc. have agreed to sell four ho...

[2] sim=0.6401 ✗ | Sci/Tech
    CA posts Q2 loss on restitution charges Computer Associates International reported a six percent increase in revenue during its second fiscal quarter,...

[3] sim=0.6395 ✓ | Business
    Oil prices climb in London, New York shut LONDON (AFP) - Oil prices rose over worries about a possible supply shortage of US heating oil stocks during...



## 6. Task 0 — Власна модель на повному датасеті

In [10]:
print(f"Розподіл категорій у тренувальному наборі:")
print(train_df['label'].map(label_names).value_counts())

Розподіл категорій у тренувальному наборі:
label
Business    30000
Sci/Tech    30000
Sports      30000
World       30000
Name: count, dtype: int64


In [11]:
# Тренуємо модель на всіх 120k документах
model_full = Doc2Vec(
    vector_size=100,
    min_count=2,
    epochs=20,
    workers=4
)
model_full.build_vocab(train_corpus)
print(f"Розмір словника: {len(model_full.wv)} слів")

model_full.train(
    train_corpus,
    total_examples=model_full.corpus_count,
    epochs=model_full.epochs
)
print("Повна модель (vector_size=100, epochs=20) навчена!")

Розмір словника: 42212 слів
Повна модель (vector_size=100, epochs=20) навчена!


## 7. Task 1 — Семантичний пошук та оцінка точності

In [12]:
def find_similar_docs(text, model, train_df, topn=5):
    """Знаходить найбільш схожі документи за вхідним текстом."""
    vec = model.infer_vector(tokenize(text))
    sims = model.dv.most_similar([vec], topn=topn)
    return [(train_df.iloc[doc_id], sim) for doc_id, sim in sims]

queries = [
    "NASA launches new space mission to explore Mars surface",
    "stock market falls recession economy GDP decline",
    "championship football world cup goal scored penalty"
]

for q in queries:
    print(f"\nЗапит: '{q}'")
    print("-" * 65)
    for row, sim in find_similar_docs(q, model_full, train_df, topn=3):
        print(f"  [{label_names[row['label']]}] sim={sim:.4f}: {row['text'][:120]}...")


Запит: 'NASA launches new space mission to explore Mars surface'
-----------------------------------------------------------------
  [Business] sim=0.8342: Stocks to Watch on Friday, September 3  INTEL CORP.&lt;A HREF="http://www.investor.reuters.com/FullQuote.aspx?ticker=INT...
  [Sports] sim=0.8074: Second-Ranked Auburn Downs Alabama TUSCALOOSA, Ala. (Sports Network) - Jason Campbell passed for 224 yards and a touchdo...
  [World] sim=0.8067: Israelis kill three Palestinians Israeli soldiers killed three Palestinian militants in the West Bank and Gaza Strip, Pa...

Запит: 'stock market falls recession economy GDP decline'
-----------------------------------------------------------------
  [Business] sim=0.6533: State: California gained 4,900 payroll jobs in September LOS ANGELES - California #39;s economy gained 4,900 payroll job...
  [Business] sim=0.6470: GE earnings increase 11 General Electric Co.s third-quarter earnings rose 11 percent,and revenue jumped 15 percent as a ...
  [

In [13]:
# Оцінка якості через nearest-neighbor classification
test_sample = test_df.sample(500, random_state=42)
correct = 0

for _, row in test_sample.iterrows():
    vec = model_full.infer_vector(tokenize(row['text']))
    sims = model_full.dv.most_similar([vec], topn=1)
    nearest_label = train_df.iloc[sims[0][0]]['label']
    if nearest_label == row['label']:
        correct += 1

accuracy = correct / len(test_sample)
print(f"Nearest-neighbor accuracy: {accuracy:.4f} ({correct}/{len(test_sample)})")
print(f"Baseline (random): 0.2500 (4 класи)")
print()
print("Висновок: doc2vec успішно вловлює семантичну схожість між документами,")
print("дозволяючи класифікувати тексти без будь-якого навчання з учителем.")

Nearest-neighbor accuracy: 0.6900 (345/500)
Baseline (random): 0.2500 (4 класи)

Висновок: doc2vec успішно вловлює семантичну схожість між документами,
дозволяючи класифікувати тексти без будь-якого навчання з учителем.
